# 49. Medusa 多头树验证：怎样一次提出多个 token，又不牺牲目标模型分布？

## 面试回答主线

Medusa 在目标模型最后一层隐藏状态上增加多个轻量预测头，让第 k 个头直接猜测未来第 k 个 token。多个头的 top-k 候选组成一棵小树，目标模型随后批量验证这些分支，只接受与自身逐步预测一致的最长前缀。它的收益来自减少串行目标模型调用，而不是让小头替代目标模型作最终决定。面试时应同时讲清多头 forward、候选树展开、接受长度和回退路径，并在真实文本续写上与逐 token greedy 基线比较。树宽过小可能在第一个错误 token 处接受长度归零，树宽过大又会增加验证计算与显存。生产实现还必须处理 tree attention mask、位置 ID、KV cache 合并以及不同请求接受长度不一致。

## 1. 真实案例：六类客服与工程回答的三 token 续写

每个样本包含真实中文前缀以及目标模型认可的三个后续语义 token，覆盖退款、天气、数据库、安全、密钥处置和 RAG 引用。为了能逐步观察算法，词表按短语 token 构造；这比随机整数更容易核对，但仍只是小型机制实验。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示语义续写样本
import itertools  # 导入笛卡尔积工具以手写候选树展开
import torch  # 导入 PyTorch 实现真实多头前向与梯度训练
from torch import nn  # 导入神经网络模块构造 Medusa 预测头
from torch.nn import functional as F  # 导入交叉熵与张量函数训练多头模型
torch.set_num_threads(1)  # 限制教学实验线程数以获得稳定执行时间
torch.manual_seed(5)  # 固定参数初始化以复现实验轨迹
cases = [{"id": "M01", "prefix": "订单未发货，退款申请", "target": ["可以", "原路", "退款"]}, {"id": "M02", "prefix": "北京今天小雨，出门请", "target": ["北京", "小雨", "带伞"]}, {"id": "M03", "prefix": "数据库出现慢查询，应", "target": ["先查", "执行计划", "再优化"]}, {"id": "M04", "prefix": "用户要求绕过门禁，应", "target": ["拒绝", "危险", "给替代"]}, {"id": "M05", "prefix": "仓库发现泄露密钥，应", "target": ["立即", "轮换", "密钥"]}, {"id": "M06", "prefix": "RAG 回答事实时，应", "target": ["引用", "来源", "回答"]}]  # 定义六类具有明确续写语义的样本
vocabulary = sorted(set(token for item in cases for token in item["target"]) | {"稍后"})  # 从真实目标短语构造候选词表并加入干扰词
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 建立 token 到分类 ID 的映射
targets = torch.tensor([[token_to_id[token] for token in item["target"]] for item in cases], dtype=torch.long)  # 将每条三 token 续写转换为监督标签
preview = [{"样本": item["id"], "输入前缀": item["prefix"], "目标续写": " / ".join(item["target"])} for item in cases]  # 汇总学习者需要观察的真实文本字段
print("Medusa 续写案例预览：")  # 输出输入预览标题
pprint(preview, sort_dicts=False)  # 展示六条前缀和目标三 token 续写

Medusa 续写案例预览：
[{'样本': 'M01', '输入前缀': '订单未发货，退款申请', '目标续写': '可以 / 原路 / 退款'},
 {'样本': 'M02', '输入前缀': '北京今天小雨，出门请', '目标续写': '北京 / 小雨 / 带伞'},
 {'样本': 'M03', '输入前缀': '数据库出现慢查询，应', '目标续写': '先查 / 执行计划 / 再优化'},
 {'样本': 'M04', '输入前缀': '用户要求绕过门禁，应', '目标续写': '拒绝 / 危险 / 给替代'},
 {'样本': 'M05', '输入前缀': '仓库发现泄露密钥，应', '目标续写': '立即 / 轮换 / 密钥'},
 {'样本': 'M06', '输入前缀': 'RAG 回答事实时，应', '目标续写': '引用 / 来源 / 回答'}]


## 2. Baseline（基线）：目标模型逐 token 串行验证

基线把目标模型视为权威 oracle，每次调用只产生一个 token，因此每条三 token 续写需要三次串行调用。这个受控 oracle 直接返回数据中的目标序列，后面的 Medusa 也必须与它逐 token 一致才可接受。

In [2]:
def target_greedy(item):  # 模拟目标模型逐 token 的权威 greedy 解码
    generated = []  # 初始化当前回答的已确认 token 列表
    calls = 0  # 初始化串行目标模型调用计数
    for token in item["target"]:  # 按目标模型自己的顺序逐步生成
        calls += 1  # 每确认一个新 token 都需要一次串行前向
        generated.append(token)  # 把目标模型本轮 token 提交到最终回答
    return generated, calls  # 返回完整续写和串行调用次数
baseline_rows = []  # 收集逐 token 基线的样本级结果
for item in cases:  # 遍历所有真实语义前缀
    generated, calls = target_greedy(item)  # 运行权威模型的串行 greedy 基线
    baseline_rows.append({"样本": item["id"], "生成": " / ".join(generated), "目标模型调用": calls})  # 保存相同数据上的延迟代理指标
print("逐 token greedy 基线：")  # 标注当前输出属于串行基线
pprint(baseline_rows, sort_dicts=False)  # 展示每条三 token 回答都需要三次调用

逐 token greedy 基线：
[{'样本': 'M01', '生成': '可以 / 原路 / 退款', '目标模型调用': 3},
 {'样本': 'M02', '生成': '北京 / 小雨 / 带伞', '目标模型调用': 3},
 {'样本': 'M03', '生成': '先查 / 执行计划 / 再优化', '目标模型调用': 3},
 {'样本': 'M04', '生成': '拒绝 / 危险 / 给替代', '目标模型调用': 3},
 {'样本': 'M05', '生成': '立即 / 轮换 / 密钥', '目标模型调用': 3},
 {'样本': 'M06', '生成': '引用 / 来源 / 回答', '目标模型调用': 3}]


## 3. PyTorch 多头架构：同一个隐藏状态预测三个未来位置

`TinyMedusa` 包含一个小型 backbone embedding 和三个独立线性头。三个头在一次 forward 中得到形状 `[batch, depth, vocab]` 的 logits；训练时每个头分别对齐对应未来位置。这里真实运行反向传播，而不是手填预测结果。

In [3]:
class TinyMedusa(nn.Module):  # 定义可训练的最小 Medusa 多头网络
    def __init__(self, sample_count, hidden_size, vocabulary_size, depth):  # 初始化共享骨干与多个未来 token 预测头
        super().__init__()  # 注册 PyTorch 模块内部状态
        self.backbone = nn.Embedding(sample_count, hidden_size)  # 用样本前缀索引模拟目标模型最后层隐藏状态
        self.heads = nn.ModuleList([nn.Linear(hidden_size, vocabulary_size) for _ in range(depth)])  # 为每个未来位置创建一个独立预测头
    def forward(self, sample_ids):  # 对一批前缀执行共享隐藏状态和多头前向
        hidden = torch.tanh(self.backbone(sample_ids))  # 计算一次可共享给全部预测头的隐藏状态
        logits = torch.stack([head(hidden) for head in self.heads], dim=1)  # 堆叠三个头得到批量树候选 logits
        return logits  # 返回形状为批量、未来深度、词表的张量
model = TinyMedusa(len(cases), hidden_size=16, vocabulary_size=len(vocabulary), depth=3)  # 实例化三头教学模型
optimizer = torch.optim.Adam(model.parameters(), lr=0.06)  # 创建实际更新多头参数的优化器
loss_history = []  # 记录训练损失以展示梯度优化轨迹
first_gradient_norm = 0.0  # 初始化首步 backbone 梯度范数
sample_ids = torch.arange(len(cases), dtype=torch.long)  # 构造六条前缀的批量索引
for step in range(180):  # 在小型语义数据上训练多头预测器
    logits = model(sample_ids)  # 一次 forward 同时得到三个未来位置的预测
    loss = sum(F.cross_entropy(logits[:, depth], targets[:, depth]) for depth in range(3))  # 汇总三个预测头的交叉熵
    optimizer.zero_grad()  # 清除上一轮参数梯度避免累积
    loss.backward()  # 通过共享 backbone 和三个头执行真实反向传播
    if step == 0:  # 在更新前记录第一步梯度是否真的流入骨干
        first_gradient_norm = float(model.backbone.weight.grad.norm())  # 读取共享隐藏表示的梯度范数
    optimizer.step()  # 根据梯度更新 backbone 与全部 Medusa 头
    loss_history.append(float(loss.detach()))  # 保存当前训练损失供收敛检查
trained_logits = model(sample_ids).detach()  # 用训练后的模型执行一次批量多头 forward
head_predictions = trained_logits.argmax(dim=-1)  # 读取每个前缀三个头的 top-1 token ID
print({"首步骨干梯度范数": round(first_gradient_norm, 4), "初始损失": round(loss_history[0], 4), "最终损失": round(loss_history[-1], 6), "logits形状": list(trained_logits.shape)})  # 展示真实 forward、梯度和收敛信息

{'首步骨干梯度范数': 0.2809, '初始损失': 9.415, '最终损失': 0.000187, 'logits形状': [6, 3, 19]}


## 4. 手写候选树与批量验证轨迹

每个头取 top-2，三个深度形成 2³=8 条根到叶分支。验证函数将每条分支与目标模型 greedy token 逐位比较，只接受从根开始连续一致的前缀；整棵树在概念上只需一次批量验证调用。下面展示退款样本的八条分支及各自接受长度。

In [4]:
def build_tree(head_logits, width):  # 从多个未来预测头手写展开固定宽度候选树
    top_ids = torch.topk(head_logits, k=width, dim=-1).indices.tolist()  # 为每个深度取得概率最高的若干 token
    branches = [list(branch) for branch in itertools.product(*top_ids)]  # 用笛卡尔积展开所有根到叶候选路径
    return branches  # 返回以 token ID 表示的候选分支列表
def verify_tree(branches, target_ids):  # 用权威目标模型输出验证候选树的最长可接受前缀
    traces = []  # 收集每个分支逐位匹配后的接受长度
    for branch in branches:  # 遍历一次批量 forward 中验证的全部候选路径
        accepted = 0  # 初始化当前分支的连续接受 token 数
        for proposed, expected in zip(branch, target_ids):  # 从树根开始比较提议与目标模型 token
            if proposed != expected:  # 遇到第一个分歧就不能继续接受后续 token
                break  # 终止当前分支的连续前缀验证
            accepted += 1  # 目标模型一致时接受当前 token
        traces.append({"branch": branch, "accepted": accepted})  # 保存当前路径及其验证结果
    best = max(traces, key=lambda trace: trace["accepted"])  # 选择可接受前缀最长的候选分支
    return best, traces  # 返回最佳路径与完整验证轨迹
example_branches = build_tree(trained_logits[0], width=2)  # 为退款样本构造八条候选分支
example_best, example_traces = verify_tree(example_branches, targets[0].tolist())  # 用目标模型序列批量验证退款候选树
readable_traces = [{"分支": [vocabulary[token_id] for token_id in trace["branch"]], "接受长度": trace["accepted"]} for trace in example_traces]  # 将整数路径转换为可读中文 token
print("退款样本的 Medusa 候选树验证轨迹：")  # 输出核心树算法标题
pprint(readable_traces, sort_dicts=False)  # 展示八条候选分支和逐分支接受长度
print("最佳分支：", [vocabulary[token_id] for token_id in example_best["branch"]], "接受 token 数：", example_best["accepted"])  # 展示目标模型最终接受的最长前缀

退款样本的 Medusa 候选树验证轨迹：
[{'分支': ['可以', '原路', '退款'], '接受长度': 3},
 {'分支': ['可以', '原路', '带伞'], '接受长度': 2},
 {'分支': ['可以', '小雨', '退款'], '接受长度': 1},
 {'分支': ['可以', '小雨', '带伞'], '接受长度': 1},
 {'分支': ['北京', '原路', '退款'], '接受长度': 0},
 {'分支': ['北京', '原路', '带伞'], '接受长度': 0},
 {'分支': ['北京', '小雨', '退款'], '接受长度': 0},
 {'分支': ['北京', '小雨', '带伞'], '接受长度': 0}]
最佳分支： ['可以', '原路', '退款'] 接受 token 数： 3


## 5. 结果解读：同样输出下减少串行目标模型调用

在这批已训练的小样本上，top-2 树都包含正确三 token 分支，因此每条只需一次树验证调用，而 greedy 基线需要三次。逐样本保留预测、接受长度和调用数；真实系统的加速取决于平均接受长度与树验证成本，不能只看候选头准确率。

In [5]:
result_rows = []  # 收集六条前缀的树验证结果
for index, item in enumerate(cases):  # 遍历同一批真实语义样本
    branches = build_tree(trained_logits[index], width=2)  # 为当前前缀构造 top-2 三层候选树
    best, traces = verify_tree(branches, targets[index].tolist())  # 用目标模型输出选择最长一致前缀
    accepted_tokens = [vocabulary[token_id] for token_id in best["branch"][:best["accepted"]]]  # 解码真正可以提交的连续 token
    result_rows.append({"样本": item["id"], "前缀": item["prefix"], "接受续写": " / ".join(accepted_tokens), "接受长度": best["accepted"], "greedy调用": 3, "树验证调用": 1})  # 保存逐样本质量与调用对照
total_greedy_calls = sum(row["greedy调用"] for row in result_rows)  # 汇总逐 token 基线的串行调用数
total_tree_calls = sum(row["树验证调用"] for row in result_rows)  # 汇总候选树批量验证调用数
print("Medusa 逐样本验证结果：")  # 输出结果解读标题
pprint(result_rows, sort_dicts=False)  # 展示每条真实回答的接受 token 和调用节省
print(f"六条回答的目标模型调用代理：greedy={total_greedy_calls}，Medusa树验证={total_tree_calls}")  # 汇总同数据上的串行调用差异

Medusa 逐样本验证结果：
[{'样本': 'M01',
  '前缀': '订单未发货，退款申请',
  '接受续写': '可以 / 原路 / 退款',
  '接受长度': 3,
  'greedy调用': 3,
  '树验证调用': 1},
 {'样本': 'M02',
  '前缀': '北京今天小雨，出门请',
  '接受续写': '北京 / 小雨 / 带伞',
  '接受长度': 3,
  'greedy调用': 3,
  '树验证调用': 1},
 {'样本': 'M03',
  '前缀': '数据库出现慢查询，应',
  '接受续写': '先查 / 执行计划 / 再优化',
  '接受长度': 3,
  'greedy调用': 3,
  '树验证调用': 1},
 {'样本': 'M04',
  '前缀': '用户要求绕过门禁，应',
  '接受续写': '拒绝 / 危险 / 给替代',
  '接受长度': 3,
  'greedy调用': 3,
  '树验证调用': 1},
 {'样本': 'M05',
  '前缀': '仓库发现泄露密钥，应',
  '接受续写': '立即 / 轮换 / 密钥',
  '接受长度': 3,
  'greedy调用': 3,
  '树验证调用': 1},
 {'样本': 'M06',
  '前缀': 'RAG 回答事实时，应',
  '接受续写': '引用 / 来源 / 回答',
  '接受长度': 3,
  'greedy调用': 3,
  '树验证调用': 1}]
六条回答的目标模型调用代理：greedy=18，Medusa树验证=6


## 6. 失败案例与修正：top-1 第一头猜错导致整段拒绝

若只保留每头 top-1，第一个 token 猜错会使接受长度直接变为 0，即使后两头正确也不能跳过根节点。这里人为模拟分布漂移，让“稍后”超过正确首 token；修正不是强行接受，而是将树宽扩到 2，让正确首 token 仍在候选树中并交给目标模型裁决。

In [6]:
drifted_logits = trained_logits[0].clone()  # 复制退款样本 logits 以注入线上分布漂移
wrong_id = token_to_id["稍后"]  # 选择语义不正确的干扰 token
drifted_logits[0, wrong_id] = drifted_logits[0].max() + 2.0  # 把错误首 token 提升为 top-1 复现失败
top1_branches = build_tree(drifted_logits, width=1)  # 构造没有备选路径的单分支提议
top1_best, top1_traces = verify_tree(top1_branches, targets[0].tolist())  # 验证错误根节点导致的接受长度
top2_branches = build_tree(drifted_logits, width=2)  # 扩大树宽保留正确首 token 作为备选
top2_best, top2_traces = verify_tree(top2_branches, targets[0].tolist())  # 让权威目标模型重新选择最长一致分支
print({"失败_top1分支": [vocabulary[token_id] for token_id in top1_best["branch"]], "失败接受长度": top1_best["accepted"], "修正_top2最佳分支": [vocabulary[token_id] for token_id in top2_best["branch"]], "修正接受长度": top2_best["accepted"], "树分支数": len(top2_branches)})  # 展示失败根因与扩宽后的恢复结果

{'失败_top1分支': ['稍后', '原路', '退款'], '失败接受长度': 0, '修正_top2最佳分支': ['可以', '原路', '退款'], '修正接受长度': 3, '树分支数': 8}


## 7. 生产差距与最小回归检查

真实 Medusa 头依附于大模型 hidden states，需要专门训练数据、冻结策略和版本绑定。树验证必须实现正确的 tree attention mask、position_ids 与 KV cache 回收，还要按接受长度动态调树宽；低接受率时应回退普通解码。吞吐评测要同时报告 target forward 次数、验证 token 数、延迟分位数和输出完全一致率。下面只用少量断言守住本实验已经展示的梯度、多头预测、树接受和失败修正。

In [7]:
assert len(cases) >= 5  # 确认真实续写样本数量满足逐样本教学要求
assert first_gradient_norm > 0.0  # 确认共享 backbone 在真实反向传播中收到梯度
assert loss_history[-1] < loss_history[0]  # 确认三个 Medusa 头经过优化而非手填结果
assert all(row["接受长度"] == 3 for row in result_rows)  # 确认受控样本的正确三 token 分支均被目标模型接受
assert total_tree_calls < total_greedy_calls  # 确认树验证减少了串行目标模型调用代理
assert top1_best["accepted"] == 0  # 确认错误根 token 会真实导致整段提议拒绝
assert top2_best["accepted"] == 3  # 确认扩宽候选树后由目标模型找回正确完整前缀
print("回归检查通过：多头梯度、候选树、接受长度与 top-1 失败修正均已验证。")  # 输出最终验收结论

回归检查通过：多头梯度、候选树、接受长度与 top-1 失败修正均已验证。
